In [1]:
%cd ..

/home/yujin109/airfoil-completion


/home/yujin109/airfoil-completion/.venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [4]:
import math
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

from src.data import get_dataloader
from src.metrics import (
    evaluate_generated_samples,
    compute_euclid_dist,
    compute_generation_diversity,
    compute_mean_deviation,
)

In [ ]:
def evaluate_dataset(device=None, eval_ratio: float = 1.0):
    """
    データセットの一部だけ評価する。
    eval_ratio: 評価に使うデータセットの割合 (0.0 < eval_ratio <= 1.0)
    """
    if not (0.0 < eval_ratio <= 1.0):
        raise ValueError("eval_ratio must be between 0.0 (exclusive) and 1.0 (inclusive)")

    # デバイス設定
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ローダーとデータセット取得（シャッフルオフ）
    loader, dataset = get_dataloader(shuffle=False)

    total_samples = len(dataset)
    max_samples = int(math.ceil(total_samples * eval_ratio))
    processed = 0
    all_metrics = []
    all_generated = []

    pbar = tqdm(loader, desc=f"Evaluating ({eval_ratio*100:.0f}%)", unit="batch")
    for coords_norm, cls_norm in pbar:
        batch_size = coords_norm.size(0)
        # 残りサンプル数
        remaining = max_samples - processed
        if remaining <= 0:
            break

        # 最後のバッチでオーバーする場合はスライス
        if batch_size > remaining:
            coords_norm = coords_norm[:remaining]
            cls_norm = cls_norm[:remaining]

        coords_norm = coords_norm.to(device)  # [B,2,248]
        cls_denorm = dataset.denormalize_cl(cls_norm)
        conditioned_cls = cls_denorm.squeeze(1).to(device)  # [B]

        metrics = evaluate_generated_samples(samples=coords_norm, conditioned_cls=conditioned_cls, dataset=dataset)
        all_metrics.append(metrics)
        processed += coords_norm.size(0)

        # プログレスバーに「処理済サンプル数／目標サンプル数」を表示
        pbar.set_postfix({"processed": f"{processed}/{max_samples}"})

        coords_denorm = dataset.denormalize_coord(coords_norm)
        all_generated.append(coords_denorm)

    all_batch = torch.cat(all_generated, dim=0)  # shape: [71*num_samples_for_each_cl, 2, 248]

    # 全体バッチでの多様度指標を一度だけ計算
    generation_diversity_all = compute_generation_diversity(all_batch)
    mu_all                = compute_mean_deviation(all_batch)
    euclid_dist_all       = compute_euclid_dist(all_batch)

    if not all_metrics:
        print("評価するサンプルがありませんでした。eval_ratio を確認してください。")
        return

    # バッチごとの結果を集約して平均
    metrics_arr = np.stack(all_metrics, axis=0)  # [num_batches, 9]
    mean_metrics = metrics_arr.mean(axis=0)

    names = [
        "convexity_loss",
        "smoothness_loss",
        "total_turning_angle",
        "smoothness_phi",
          "MSE (CL error)",
        "MAPE (CL error)",
        "convergence_ratio",
        "strict_convergence_ratio",
        "diversity (L2 norm²)",
        "mean_deviation (μ)",
        "euclid_dist_mean",
    ]

    print("\n===== 全データセット評価結果 =====")
    for name, val in zip(names, mean_metrics):
        print(f"{name: <30}: {val:.6f}")
    print(f"generation_diversity_all: {generation_diversity_all}")
    print(f"mu_all: {mu_all}")
    print(f"euclid_dist_all: {euclid_dist_all}")


if __name__ == "__main__":
    evaluate_dataset(eval_ratio=1)

Evaluating (100%):   0%|          | 0/118 [00:00<?, ?batch/s]

Evaluating (100%): 100%|██████████| 118/118 [07:54<00:00,  4.02s/batch, processed=3767/3767]


===== 全データセット評価結果 =====
convexity_loss                : 0.019577
smoothness_loss               : 0.001495
total_turning_angle           : 2.000000
MSE (CL error)                : 0.000001
MAPE (CL error)               : 0.002017
convergence_ratio             : 0.993644
strict_convergence_ratio      : 0.993644
diversity (L2 norm²)          : 89.183734
mean_deviation (μ)            : 7.465275
euclid_dist_mean              : 1.475445
generation_diversity_all: 1.2737970352172852
mu_all: 1.1199133396148682
euclid_dist_all: 0.01838875475124747


In [ ]:
===== 全データセット評価結果 =====
convexity_loss                : 0.019577
smoothness_loss               : 0.001495
total_turning_angle           : 2.000000
MSE (CL error)                : 0.000001
MAPE (CL error)               : 0.002017
convergence_ratio             : 0.993644
strict_convergence_ratio      : 0.993644
diversity (L2 norm²)          : 89.183734
mean_deviation (μ)            : 7.465275
euclid_dist_mean              : 1.475445
generation_diversity_all: 1.2737970352172852
mu_all: 1.1199133396148682
euclid_dist_all: 0.01838875475124747

In [ ]:
convexity_loss_mean: 0.023404117673635483
smoothness_loss_mean: 0.0014954852991885415
total_turning_angle: 2.0084507286903013
cl_mse_mean: 0.00025701249920265056
cl_mape_mean: 1.4702189426105048
cl_convergence_ratio_raw: 0.9281690140845071
cl_convergence_ratio_strict: 0.9281690140845071
generation_diversity_mean: 366.6954618104747
mu: 17.82558201400327
euclid_dist_mean: 5.969959363803058
generation_diversity_all: 483.59783935546875
mu_all: 21.42081642150879
euclid_dist_all: 0.8253019950759243

In [ ]:
convexity_loss_mean: 0.023316174745559692
smoothness_loss_mean: 0.0014953446971266993
total_turning_angle: 2.011267632314914
cl_mse_mean: 0.00034408366509723626
cl_mape_mean: 1.6427255464514459
cl_convergence_ratio_raw: 0.9253521126760563
cl_convergence_ratio_strict: 0.9253521126760563
generation_diversity_mean: 382.91990081357284
mu: 18.288531827255035
euclid_dist_mean: 6.096686787672446